# Chapter 1.1 — Reddit: Human Emotional Debris

**Dataset 1 of 3.** This notebook scrapes the public Reddit JSON endpoint to collect the textual residue of three communities that, between them, describe the kind of digital remains this project is about:

- **r/GriefSupport** — the raw register of grief. People writing to no one in particular about people who are not there.
- **r/lostmedia** — people actively searching for fragments of media that have disappeared from the internet. A community whose entire activity is *the recovery of digital remains*. Conceptually this is the most on-the-nose subreddit for the project.
- **r/LetterToMyEx** — unsent messages to people who will not read them. Half-finished sentences, in the project's own thesis language.

**Target:** ≈ 90 posts per subreddit → ≥ 250 total (assignment minimum is 200–300).

**Method:** Reddit's public JSON endpoint (no auth required) sorted by `top` of `all` time, paginated via the `after` token. Polite User-Agent + 2 s sleep between requests. Posts whose `selftext` is empty / `[removed]` / `[deleted]` are dropped (we need text for vectorisation).

**Output:** `data/raw/reddit/reddit_posts.csv` with columns: `id, subreddit, title, selftext, score, num_comments, created_utc, permalink, author, url`.

In [2]:
import sys, os, time, json, datetime as dt
from pathlib import Path

# Robust project-root finder: works whether JupyterLab launched the kernel
# from the project folder or from anywhere else on disk.
def _find_project_root(marker="sa_utils.py"):
    p = Path.cwd().resolve()
    for c in [p] + list(p.parents):
        if (c / marker).exists(): return c
    cowork = Path.home() / "Library/Application Support/Claude/local-agent-mode-sessions"
    if cowork.exists():
        for hit in cowork.rglob(marker):
            return hit.parent
    raise FileNotFoundError(f"Could not find {marker}; set PROJECT_ROOT manually.")
PROJECT_ROOT = _find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from sa_utils import load_env, make_session, polite_sleep, DATA_RAW
import pandas as pd

load_env()
OUT_DIR = DATA_RAW / "reddit"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Output dir:   {OUT_DIR}")

.env loaded. OPENAI_API_KEY present: True
Project root: /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs
Output dir:   /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/reddit


## Fetcher

We paginate through `top.json?t=all` using the `after` token Reddit returns at the bottom of each page. Reddit caps `limit` at 100 per call.

Failure handling: 4xx/5xx and 429s are retried by the urllib3 adapter inside `make_session()`. If we get fewer posts than `target` (because the subreddit doesn't have that many), we accept whatever we got.

In [3]:
def fetch_subreddit_top(session, subreddit: str, target: int = 90, time_range: str = "all", sleep_s: float = 2.0):
    """Pull up to `target` 'top' posts from a subreddit via the JSON endpoint."""
    url = f"https://www.reddit.com/r/{subreddit}/top.json"
    posts, after = [], None
    page = 0
    while len(posts) < target:
        page += 1
        params = {"limit": 100, "t": time_range}
        if after:
            params["after"] = after
        r = session.get(url, params=params, timeout=20)
        if r.status_code != 200:
            print(f"  [{subreddit}] page {page}: HTTP {r.status_code} — stopping")
            break
        data = r.json().get("data", {})
        children = data.get("children", [])
        if not children:
            print(f"  [{subreddit}] page {page}: no more posts — stopping")
            break
        for c in children:
            d = c.get("data", {})
            posts.append({
                "id":          d.get("id"),
                "subreddit":   d.get("subreddit"),
                "title":       d.get("title", "") or "",
                "selftext":    d.get("selftext", "") or "",
                "score":       d.get("score", 0),
                "num_comments":d.get("num_comments", 0),
                "created_utc": d.get("created_utc"),
                "permalink":   "https://www.reddit.com" + (d.get("permalink") or ""),
                "author":      d.get("author"),
                "url":         d.get("url"),
            })
        after = data.get("after")
        print(f"  [{subreddit}] page {page}: +{len(children)} (running total {len(posts)})")
        if not after:
            break
        polite_sleep(sleep_s)
    return posts[:target]

session = make_session()
print("Session ready.")

Session ready.


In [4]:
# ---- Pull the three subreddits ----
SUBREDDITS = {
    "GriefSupport":   90,
    "lostmedia":      90,
    "LetterToMyEx":   90,
}

all_posts = []
for sub, target in SUBREDDITS.items():
    print(f"\n→ r/{sub} (target {target})")
    all_posts.extend(fetch_subreddit_top(session, sub, target=target))

print(f"\nTOTAL RAW POSTS: {len(all_posts)}")


→ r/GriefSupport (target 90)
  [GriefSupport] page 1: +100 (running total 100)

→ r/lostmedia (target 90)
  [lostmedia] page 1: +100 (running total 100)

→ r/LetterToMyEx (target 90)
  [LetterToMyEx] page 1: +5 (running total 5)

TOTAL RAW POSTS: 185


In [5]:
# ---- Filter: we need text for vectorisation, so drop empty / removed / deleted bodies ----
import re

BAD_BODIES = {"", "[removed]", "[deleted]", ".", "deleted", "removed"}

df = pd.DataFrame(all_posts).drop_duplicates(subset="id")
before = len(df)
df["selftext"] = df["selftext"].str.strip()
df = df[~df["selftext"].isin(BAD_BODIES)]
df = df[df["selftext"].str.len() >= 40]   # need at least a sentence or two
after = len(df)
print(f"Filtered {before - after} empty/removed/short posts. Remaining: {after}")

# Add a single combined text field that downstream notebooks will tokenise.
df["combined_text"] = (df["title"].fillna("") + "\n\n" + df["selftext"].fillna("")).str.strip()

# Convert created_utc to ISO for readability.
df["created_iso"] = pd.to_datetime(df["created_utc"], unit="s", utc=True).dt.strftime("%Y-%m-%d")
df = df.reset_index(drop=True)
df.head(3)

Filtered 73 empty/removed/short posts. Remaining: 112


,id,subreddit,title,selftext,score,num_comments,created_utc,permalink,author,url,combined_text,created_iso
0,1hf2kky,GriefSupport,Comic I made following my brother’s recent sui...,First time poster here.. I’m a cartoonist and ...,5132,253,1.734297e+09,https://www.reddit.com/r/GriefSupport/comments...,[deleted],https://www.reddit.com/gallery/1hf2kky,Comic I made following my brother’s recent sui...,2024-12-15
1,1n58dzg,GriefSupport,I lost my son in a school shooting years ago b...,Lost him before Christmas in 2012 and he was o...,4096,298,1.756680e+09,https://www.reddit.com/r/GriefSupport/comments...,No-Preparation3572,https://i.redd.it/du2blwdhofmf1.jpeg,I lost my son in a school shooting years ago b...,2025-08-31
2,1jebspe,GriefSupport,My husband was an amazing man,My husband was an amazing man. He was everythi...,2405,147,1.742322e+09,https://www.reddit.com/r/GriefSupport/comments...,aBaKePoTaTo,https://www.reddit.com/gallery/1jebspe,My husband was an amazing man\n\nMy husband wa...,2025-03-18


In [6]:
# ---- Save CSV + a raw JSON dump (handy for re-processing without re-scraping) ----
csv_path = OUT_DIR / "reddit_posts.csv"
json_path = OUT_DIR / "reddit_posts_raw.json"

df.to_csv(csv_path, index=False, encoding="utf-8")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(all_posts, f, ensure_ascii=False, indent=2)

print(f"✓ Saved {len(df)} posts to {csv_path}")
print(f"✓ Saved raw dump to {json_path}")

✓ Saved 112 posts to /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/reddit/reddit_posts.csv
✓ Saved raw dump to /Users/rao/Library/Application Support/Claude/local-agent-mode-sessions/6b9b214c-10b0-4e49-8804-78fee7ba4cfe/65bc0ee0-83a2-418a-80ae-8b2eb8d6a0cd/local_cc966565-ccf9-44a8-bd4f-b759a867e830/outputs/data/raw/reddit/reddit_posts_raw.json


## Quick stats (sanity check)

If the per-subreddit count is far below target, the subreddit may be smaller than expected, the post body filter may have been too aggressive, or Reddit may have rate-limited us. Re-run with a longer `sleep_s` if needed.

In [7]:
print("Per-subreddit counts (after filtering):")
print(df["subreddit"].value_counts().to_string())
print()
print("Token length summary (whitespace tokens):")
tok_lens = df["combined_text"].str.split().str.len()
print(tok_lens.describe().round(1).to_string())
print()
print("Date range:")
print(f"  earliest: {df['created_iso'].min()}")
print(f"  latest:   {df['created_iso'].max()}")

Per-subreddit counts (after filtering):
subreddit
GriefSupport    76
lostmedia       31
lettertomyex     5

Token length summary (whitespace tokens):
count     112.0
mean      274.3
std       249.8
min        20.0
25%       110.5
50%       185.0
75%       360.5
max      1521.0

Date range:
  earliest: 2020-12-23
  latest:   2026-04-15


## Notes for the report

When writing the dataset section in the final PDF, document:

1. **Why these three subreddits and not others.** They form a thematic triangle of the kinds of digital remains the thesis names: emotional residue (GriefSupport), the active recovery of lost media (lostmedia), and unsent words to absent recipients (LetterToMyEx).
2. **Why `top` / `all`.** Top-of-all-time concentrates the strongest emotional posts. New / hot would give us recency bias and lower median quality.
3. **Why we drop `[removed]` and `[deleted]`.** These are themselves digital remains — ironically the most thesis-aligned content — but they carry zero text for vectorisation. The fact that they exist (and how many we dropped) should be acknowledged in the writeup as an observable limit of the data: *the deepest layer of the archive is also the most illegible*.
4. **Robustness.** The Reddit JSON endpoint is a snapshot of a non-stable interface; if re-run later the counts will differ. This is the same caveat the Turin case study makes about its British Pathé scrape, and it is correct to acknowledge it here.